In [6]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import random
from torch.optim import LBFGS
from tqdm import tqdm
from model_components.models import PINNs
from model_components.util import *
import optuna
import scipy.io
seed = 0
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
step_size = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
res, b_left, b_right, b_upper, b_lower = get_data([0,2*np.pi], [0,1], 101, 101)
res_test, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101)


def set_seed(seed=0):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
smallest_rl1 = 1e10
    

mat = scipy.io.loadmat('./convection.mat')
u_true = mat['u'].reshape(101,101)

res = torch.tensor(res, dtype=torch.float32, requires_grad=True).to(device)
b_left = torch.tensor(b_left, dtype=torch.float32, requires_grad=True).to(device)
b_right = torch.tensor(b_right, dtype=torch.float32, requires_grad=True).to(device)
b_upper = torch.tensor(b_upper, dtype=torch.float32, requires_grad=True).to(device)
b_lower = torch.tensor(b_lower, dtype=torch.float32, requires_grad=True).to(device)

x_res, t_res = res[:,0:1], res[:,1:2]
x_left, t_left = b_left[:,0:1], b_left[:,1:2]
x_right, t_right = b_right[:,0:1], b_right[:,1:2]
x_upper, t_upper = b_upper[:,0:1], b_upper[:,1:2]
x_lower, t_lower = b_lower[:,0:1], b_lower[:,1:2]

res_test = torch.tensor(res_test, dtype=torch.float32, requires_grad=True).to(device)
x_test, t_test = res_test[:,0:1], res_test[:,1:2]

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform(m.weight)
        m.bias.data.fill_(0.01)
        
def objective(trial):
    global device, x_res, t_res, x_left, t_left, x_right, t_right, x_upper, t_upper, x_lower, t_lower, x_test, t_test, u_true, D1, D2, D3, kernel_size, smallest_rl1
    set_seed(0)
    
    
    d_hidden = trial.suggest_categorical('d_hidden', [128, 256, 512, 768])
    num_layer = trial.suggest_int('num_layer', 2, 6)

    model = PINNs(in_dim=2, hidden_dim=d_hidden, out_dim=1, num_layer=num_layer).to(device)

    # model.apply(init_weights)
    optim = LBFGS(model.parameters(), line_search_fn='strong_wolfe')

    n_params = get_n_params(model)
    kernel_size = 300

    D1 = kernel_size
    D2 = len(x_left)
    D3 = len(x_lower)

    def compute_ntk(J1, J2):
        Ker = torch.matmul(J1, torch.transpose(J2, 0, 1))
        return Ker


    w1, w2, w3 = 1, 1, 1

    for i in tqdm(range(1000)):
        
        if i % 20 == 0:
            J1 = torch.zeros((D1, n_params))
            J2 = torch.zeros((D2, n_params))
            J3 = torch.zeros((D3, n_params))

            batch_ind = np.random.choice(len(x_res), kernel_size, replace=False)
            x_train, t_train = x_res[batch_ind], t_res[batch_ind]

            pred_res = model(x_train, t_train)
            pred_left = model(x_left, t_left)
            pred_upper = model(x_upper, t_upper)
            pred_lower = model(x_lower, t_lower)

            for j in range(len(x_train)):
                model.zero_grad()
                pred_res[j,0].backward(retain_graph=True)
                J1[j, :] = torch.cat([
                    p.grad.view(-1) if p.grad is not None else torch.zeros_like(p).view(-1) 
                    for p in model.parameters()
                    ])


            for j in range(len(x_left)):
                model.zero_grad()
                pred_left[j,0].backward(retain_graph=True)
                J2[j, :] = torch.cat([
                    p.grad.view(-1) if p.grad is not None else torch.zeros_like(p).view(-1) 
                    for p in model.parameters()
                    ])

            for j in range(len(x_lower)):
                model.zero_grad()
                pred_lower[j,0].backward(retain_graph=True)
                pred_upper[j,0].backward(retain_graph=True)
                J3[j, :] = torch.cat([
                    p.grad.view(-1) if p.grad is not None else torch.zeros_like(p).view(-1) 
                    for p in model.parameters()
                    ])

            K1 = torch.trace(compute_ntk(J1, J1))
            K2 = torch.trace(compute_ntk(J2, J2))
            K3 = torch.trace(compute_ntk(J3, J3))
            
            K = K1+K2+K3

            w1 = K.item() / K1.item()
            w2 = K.item() / K2.item()
            w3 = K.item() / K3.item()
        def closure():
            pred_res = model(x_res, t_res)
            pred_left = model(x_left, t_left)
            pred_right = model(x_right, t_right)
            pred_upper = model(x_upper, t_upper)
            pred_lower = model(x_lower, t_lower)

            u_x = torch.autograd.grad(pred_res, x_res, grad_outputs=torch.ones_like(pred_res), retain_graph=True, create_graph=True)[0]
            u_t = torch.autograd.grad(pred_res, t_res, grad_outputs=torch.ones_like(pred_res), retain_graph=True, create_graph=True)[0]

            loss_res = torch.mean((u_t + 50 * u_x) ** 2)
            loss_bc = torch.mean((pred_upper - pred_lower) ** 2)
            loss_ic = torch.mean((pred_left[:,0] - torch.sin(x_left[:,0])) ** 2)


            loss = w1 * loss_res + w2 * loss_bc + w3 * loss_ic
            optim.zero_grad()
            loss.backward()
            return loss
        
        optim.step(closure)



    with torch.no_grad():
        pred = model(x_test, t_test)[:,0:1]
        pred = pred.cpu().detach().numpy()

    pred = pred.reshape(101,101)


    rl1 = np.sum(np.abs(u_true-pred)) / np.sum(np.abs(u_true))
    
    if rl1 < smallest_rl1:
        smallest_rl1 = rl1
        print(f"New best model found with RL1: {smallest_rl1}")
        torch.save(model.state_dict(), f'saves/conv-pinn-{trial.number}.pth')
        
    return rl1
        

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

print('Best trial:')
trial = study.best_trial
print(f'  Value: {trial.value}')
print('  Params:')
for key, val in trial.params.items():
    print(f'    {key}: {val}')


[I 2025-08-19 18:19:18,164] A new study created in memory with name: no-name-7cf09f03-69c6-4a75-b741-01e6269dc0d2
100%|██████████| 1000/1000 [09:18<00:00,  1.79it/s]
[I 2025-08-19 18:28:36,541] Trial 0 finished with value: 0.8366573738721995 and parameters: {'d_hidden': 512, 'num_layer': 4}. Best is trial 0 with value: 0.8366573738721995.


New best model found with RL1: 0.8366573738721995


100%|██████████| 1000/1000 [04:31<00:00,  3.68it/s]
[I 2025-08-19 18:33:08,145] Trial 1 finished with value: 0.8910319222959594 and parameters: {'d_hidden': 256, 'num_layer': 3}. Best is trial 0 with value: 0.8366573738721995.
100%|██████████| 1000/1000 [05:19<00:00,  3.13it/s]
[I 2025-08-19 18:38:27,225] Trial 2 finished with value: 0.6639772700730778 and parameters: {'d_hidden': 128, 'num_layer': 5}. Best is trial 2 with value: 0.6639772700730778.


New best model found with RL1: 0.6639772700730778


100%|██████████| 1000/1000 [09:10<00:00,  1.81it/s]
[I 2025-08-19 18:47:38,312] Trial 3 finished with value: 0.8922716337199762 and parameters: {'d_hidden': 768, 'num_layer': 3}. Best is trial 2 with value: 0.6639772700730778.
100%|██████████| 1000/1000 [04:32<00:00,  3.68it/s]
[I 2025-08-19 18:52:10,436] Trial 4 finished with value: 0.8910319222959594 and parameters: {'d_hidden': 256, 'num_layer': 3}. Best is trial 2 with value: 0.6639772700730778.
100%|██████████| 1000/1000 [04:59<00:00,  3.34it/s]
[I 2025-08-19 18:57:10,018] Trial 5 finished with value: 0.9396688525018282 and parameters: {'d_hidden': 128, 'num_layer': 3}. Best is trial 2 with value: 0.6639772700730778.
100%|██████████| 1000/1000 [08:56<00:00,  1.87it/s]
[I 2025-08-19 19:06:06,267] Trial 6 finished with value: 0.745694344210413 and parameters: {'d_hidden': 512, 'num_layer': 5}. Best is trial 2 with value: 0.6639772700730778.
100%|██████████| 1000/1000 [03:13<00:00,  5.17it/s]
[I 2025-08-19 19:09:19,868] Trial 7 finis

New best model found with RL1: 0.6627952322303342


100%|██████████| 1000/1000 [04:58<00:00,  3.36it/s]
[I 2025-08-19 19:51:11,040] Trial 12 finished with value: 0.6627952322303342 and parameters: {'d_hidden': 128, 'num_layer': 6}. Best is trial 11 with value: 0.6627952322303342.
100%|██████████| 1000/1000 [04:58<00:00,  3.35it/s]
[I 2025-08-19 19:56:09,218] Trial 13 finished with value: 0.6627952322303342 and parameters: {'d_hidden': 128, 'num_layer': 6}. Best is trial 11 with value: 0.6627952322303342.
100%|██████████| 1000/1000 [04:58<00:00,  3.35it/s]
[I 2025-08-19 20:01:07,755] Trial 14 finished with value: 0.6627952322303342 and parameters: {'d_hidden': 128, 'num_layer': 6}. Best is trial 11 with value: 0.6627952322303342.
100%|██████████| 1000/1000 [04:57<00:00,  3.36it/s]
[I 2025-08-19 20:06:05,134] Trial 15 finished with value: 0.6627952322303342 and parameters: {'d_hidden': 128, 'num_layer': 6}. Best is trial 11 with value: 0.6627952322303342.
100%|██████████| 1000/1000 [15:57<00:00,  1.04it/s]
[I 2025-08-19 20:22:03,252] Tria

Best trial:
  Value: 0.6627952322303342
  Params:
    d_hidden: 128
    num_layer: 6
